# Creación de Transformadores y Pipelines personalizados

En este notebook se muestra la creación de transformadores y Pipelines personalizados.

## Conjunto de datos

### Descripción
NSL-KDD es un conjunto de datos propuesto para resolver algunos de los problemas inherentes del conjunto de datos KDD'99. Sigue siendo un benchmark efectivo para comparar métodos de detección de intrusiones.

### Ficheros de datos utilizados
* **KDDTrain+.txt**: Conjunto de entrenamiento completo NSL-KDD en formato TXT

### Referencias
_M. Tavallaee, E. Bagheri, W. Lu, and A. Ghorbani, "A Detailed Analysis of the KDD CUP 99 Data Set," Submitted to Second IEEE Symposium on Computational Intelligence for Security and Defense Applications (CISDA), 2009._

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

## Funciones auxiliares

In [ ]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty_level'
]

NORMAL_LABEL = 'normal'


def load_kdd_dataset(data_path):
    """Lectura del conjunto de datos NSL-KDD desde TXT."""
    df = pd.read_csv(data_path, header=None, names=COLUMN_NAMES)
    df.drop('difficulty_level', axis=1, inplace=True)
    df['class'] = df['class'].apply(lambda x: NORMAL_LABEL if x == NORMAL_LABEL else 'anomaly')
    for col in df.select_dtypes(exclude=['object', 'str']).columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

In [ ]:
def train_val_test_split(df, rstate=42, shuffle=True, stratify=None):
    strat = df[stratify] if stratify else None
    train_set, test_set = train_test_split(
        df, test_size=0.4, random_state=rstate, shuffle=shuffle, stratify=strat)
    strat = test_set[stratify] if stratify else None
    val_set, test_set = train_test_split(
        test_set, test_size=0.5, random_state=rstate, shuffle=shuffle, stratify=strat)
    return (train_set, val_set, test_set)

## Lectura del conjunto de datos

In [ ]:
df = load_kdd_dataset("NSL_KDD-master/KDDTrain+.txt")

In [ ]:
df

## División del conjunto de datos

In [ ]:
train_set, val_set, test_set = train_val_test_split(df, stratify='protocol_type')

In [ ]:
print("Longitud del Training Set:", len(train_set))
print("Longitud del Validation Set:", len(val_set))
print("Longitud del Test Set:", len(test_set))

## API sklearn

Antes de continuar vamos a hacer una pequeña reseña sobre como funcionan las APIs de sklearn:
* **Estimators**: Cualquier objeto que puede estimar algún parámetro:
    * El propio estimator se forma mediante el método `fit()`, que siempre toma un dataset como argumento
    * Cualquier otro parámetro de este método, es un hiperparámetro
* **Transformers**: Son estimadores capaces de transformar el conjunto de datos (como Imputer)
    * La transformación se realiza mediante el método `transform()`
    * Reciben un dataset como parámetro de entrada
* **Predictors**: Son estimadores capaces de realizar predicciones
    * La predicción se realiza mediante el método `predict()`
    * Reciben un dataset como entrada
    * Retornan un dataset con las predicciones
    * Tienen un método `score()` para evaluar el resultado de la predicción

## 1. Construyendo transformadores personalizados

La creación de transformadores propios permite mantener el código mucho más limpio y estructurado a la hora de preparar los datos para los algoritmos de ML. Además, facilitan la reutilización de código para otros proyectos.

Antes de comenzar, vamos a recuperar el conjunto de datos limpio y vamos a separar las etiquetas del resto de los datos, no necesariamente queremos aplicar las mismas transformaciones en ambos conjuntos.

In [ ]:
X_train = train_set.drop("class", axis=1)
y_train = train_set["class"].copy()

In [ ]:
# Para ilustrar esta sección vamos a añadir algunos valores nulos
# a algunas características del conjunto de datos
X_train = X_train.copy()
X_train.loc[(X_train["src_bytes"] > 400) & (X_train["src_bytes"] < 800), "src_bytes"] = np.nan
X_train.loc[(X_train["dst_bytes"] > 500) & (X_train["dst_bytes"] < 2000), "dst_bytes"] = np.nan

In [ ]:
X_train

### Transformadores para atributos numéricos

In [ ]:
# Transformador creado para eliminar las filas con valores nulos
from sklearn.base import BaseEstimator, TransformerMixin

class DeleteNanRows(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        return X.dropna()

In [ ]:
delete_nan = DeleteNanRows()
X_train_prep = delete_nan.fit_transform(X_train)

In [ ]:
X_train_prep

In [ ]:
# Transformador diseñado para escalar de manera sencilla únicamente unas columnas seleccionadas
class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, attributes):
        self.attributes = attributes
    def fit(self, X, y=None):
        return self
    def transform(self, X, y=None):
        X_copy = X.copy()
        scale_attrs = X_copy[self.attributes]
        robust_scaler = RobustScaler()
        X_scaled = robust_scaler.fit_transform(scale_attrs)
        X_scaled = pd.DataFrame(X_scaled, columns=self.attributes, index=X_copy.index)
        for attr in self.attributes:
            X_copy[attr] = X_scaled[attr]
        return X_copy

In [ ]:
custom_scaler = CustomScaler(["src_bytes", "dst_bytes"])
X_train_prep = custom_scaler.fit_transform(X_train_prep)

In [ ]:
X_train.head(10)

In [ ]:
X_train_prep.head(20)

In [ ]:
# Transformador para codificar únicamente las columnas categóricas y devolver un DataFrame
from sklearn.preprocessing import OneHotEncoder

class CustomOneHotEncoding(BaseEstimator, TransformerMixin):
    def __init__(self):
        self._oh = OneHotEncoder(sparse_output=False)
        self._columns = None
    def fit(self, X, y=None):
        X_cat = X.select_dtypes(include=['object', 'str'])
        self._columns = pd.get_dummies(X_cat).columns
        self._oh.fit(X_cat)
        return self
    def transform(self, X, y=None):
        X_copy = X.copy()
        X_cat = X_copy.select_dtypes(include=['object', 'str'])
        X_cat_oh = self._oh.transform(X_cat)
        X_cat_oh = pd.DataFrame(X_cat_oh,
                                columns=self._columns,
                                index=X_copy.index)
        X_copy.drop(list(X_cat), axis=1, inplace=True)
        return X_copy.join(X_cat_oh)

## 6. Construyendo Pipelines personalizados

Las pipelines nos permiten agrupar en un flujo de ejecución todas las operaciones de transformación que necesitemos realizar sobre el conjunto de datos, esto facilita muchísimo las transformaciones para diferentes conjuntos de datos. Las cosas a tener en cuenta de estas estructuras son:
* Recibe un conjunto de pares (nombre, estimador)
* Todos menos el último deben ser transformadores
* El pipeline expone los mismos métodos que **el último estimador**
* Cuando se llama al método `fit()` del pipeline, se llama secuencialmente al método `fit_transform()` de los estimadores y se les pasa de manera secuencial el output del anterior como input del siguiente. El último invoca el método `fit()`

In [ ]:
# Construcción de un pipeline para los atributos numéricos
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer

num_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median")),
        ('rbst_scaler', RobustScaler()),
    ])

In [ ]:
# La clase imputer no admite valores categóricos, eliminamos los atributos categóricos
X_train_num = X_train.select_dtypes(exclude=['object', 'str'])

X_train_prep = num_pipeline.fit_transform(X_train_num)
X_train_prep = pd.DataFrame(X_train_prep, columns=X_train_num.columns, index=X_train_num.index)

In [ ]:
X_train_num.head(10)

In [ ]:
X_train_prep.head(10)

A continuación se presenta el método `ColumnTransformer`, que **ejecuta todos los pipelines en paralelo y concatena el resultado**. Para ello el resultado de los pipelines debe ser un valor numérico.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_attribs = list(X_train.select_dtypes(exclude=['object', 'str']))
cat_attribs = list(X_train.select_dtypes(include=['object', 'str']))

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

In [ ]:
X_train_prep = full_pipeline.fit_transform(X_train)

In [ ]:
X_train_prep = pd.DataFrame(X_train_prep, columns=list(pd.get_dummies(X_train)), index=X_train.index)

In [ ]:
X_train_prep.head(10)

In [ ]:
X_train.head(10)